In [ ]:
from osgeo import gdal
import pandas as pd
import geopandas as gpd
from tqdm import tqdm
import tempfile
import subprocess
import os

In [ ]:
def reproject_image(input_file, output_file, crs):
    ### change the crs of image into wgs84
    # Convert paths to strings for subprocess
    input_file = str(input_file)
    output_file = str(output_file)
    
    command = [
        'gdalwarp',
        '-t_srs', crs,
        input_file,
        output_file
    ]
    
    result = subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    
    if result.returncode != 0:
        print(f"Error during reprojection: {result.stderr}")
        
    return None

def crop_houses(dataset, footprints, image_bbox, input_file, output_path):
    ### Use house boundary footprints to crop houses out in each image 
    for index, footprint in footprints.iterrows(): 
        house_id = footprint['OBJECTID']
        
        with tempfile.NamedTemporaryFile(delete=True, suffix='.geojson') as tmp:
            # Save the current footprint to the temporary file
            gpd.GeoDataFrame([footprint], crs=footprints.crs).to_file(tmp.name, driver='GeoJSON')

            output_file = output_path + f'house_{house_id}.jpg'
             
            # Crop the satellite image to this single footprint
            gdal.Warp(output_file, input_file, cutlineDSName=tmp.name, cropToCutline=True, format='JPEG')

            # Delete the auxiliary .aux.xml file
            aux_file = output_file + '.aux.xml'
            if os.path.exists(aux_file):
                os.remove(aux_file)
    return None

# Crop the houses out

In [ ]:
year = 2022
crs = 'EPSG:3857'

input_path = f'F:/UofT_CoT_Orthos_Request/CityOfTorontoOrthos_{year}_TorontoCD/'
output_path = f'F:/Houses/{year}/'
image_check = f'F:/Images Check/{year}_image_check.xlsx'

footprints = gpd.read_file('C:/Users/Lenovo/Desktop/python/ra/Solar-Panel/testing/boundaries/edited boundary/PROPERTY_BOUNDARIES_WGS84_modified.shp')
image_info = pd.read_excel(image_check)
image_info = image_info.dropna()

In [ ]:
for index, image in tqdm(image_info.iterrows()):
    input_file = input_path + image['Name']
    output_file = output_path + image['Name']

    # Reproject the image into EPSG:3857
    reproject_image(input_file, output_file, crs)

    dataset = gdal.Open(output_file, gdal.GA_ReadOnly)

    houses = image['House ids'].split(',')
    footprint = footprints[footprints['OBJECTID'].isin(houses)]
    crop_houses(dataset, footprint, image['Boundary'], output_file, output_path)

    dataset = None

    # Delete the reprojected file
    os.remove(output_file)
           
    # break